# 1. 데이터 로드 및 기본 탐색

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, average_precision_score

# 데이터 로드
df = pd.read_csv('creditcard.csv')
# 데이터 구조 확인
print(df.head()) # 데이터 상단부 확인
print(df.info()) # 결측치 및 데이터 타입 확인
print(df.describe()) # 통계적 분포 확인
# 정상 거래와 사기 거래 건수 및 비율 확인
print(df['Class'].value_counts())

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

# 2. 샘플링

In [2]:
# 샘플링(정상 거래 10,000건 + 사기 거래 전체)
# 사기 거래(Class=1) 데이터는 전부 유지
fraud = df[df['Class'] == 1]

# 정상 거래(Class=0) 데이터는 10,000건만 무작위로 샘플링
# 이때 random_state는 42로 설정
normal = df[df['Class'] == 0].sample(n=10000, random_state=42)

# 두 데이터셋을 합쳐 새로운 분석용 데이터프레임(df_sampled)을 생성
df_sampled = pd.concat([fraud, normal])

# 샘플링 후의 Class별 건수와 비율을 다시 출력하여 확인
print("--- 샘플링 후 Class별 건수 ---")
print(df_sampled['Class'].value_counts())

print("\n--- 샘플링 후 Class별 비율 ---")
print(df_sampled['Class'].value_counts(normalize=True))

--- 샘플링 후 Class별 건수 ---
Class
0    10000
1      492
Name: count, dtype: int64

--- 샘플링 후 Class별 비율 ---
Class
0    0.953107
1    0.046893
Name: proportion, dtype: float64


# 3. 데이터 전처리

In [3]:
from sklearn.preprocessing import StandardScaler

# Amount 변수 표준화 (StandardScaler 사용)
scaler = StandardScaler()
# Amount_Scaled 변수 생성
df_sampled['Amount_Scaled'] = scaler.fit_transform(df_sampled[['Amount']])

# 원본 Amount 변수 제거
df_sampled.drop('Amount', axis=1, inplace=True)

# X(특성), y(타겟)로 데이터프레임 분리
X = df_sampled.drop('Class', axis=1)
y = df_sampled['Class']

# 전처리 결과 확인
print("--- 전처리 후 컬럼 목록 ---")
print(X.columns)
print("\n--- X 데이터 상단 5행 ---")
print(X.head())

--- 전처리 후 컬럼 목록 ---
Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28',
       'Amount_Scaled'],
      dtype='object')

--- X 데이터 상단 5행 ---
        Time        V1        V2        V3        V4        V5        V6  \
541    406.0 -2.312227  1.951992 -1.609851  3.997906 -0.522188 -1.426545   
623    472.0 -3.043541 -3.157307  1.088463  2.288644  1.359805 -1.064823   
4920  4462.0 -2.303350  1.759247 -0.359745  2.330243 -0.821628 -0.075788   
6108  6986.0 -4.397974  1.358367 -2.592844  2.679787 -1.128131 -1.706536   
6329  7519.0  1.234235  3.019740 -4.304597  4.732795  3.624201 -1.357746   

            V7        V8        V9  ...       V20       V21       V22  \
541  -2.537387  1.391657 -2.770089  ...  0.126911  0.517232 -0.035049   
623   0.325574 -0.067794 -0.270953  ...  2.102339  0.661696  0.435477   
4920  0.562320 -0.39

# 4. 학습 데이터와 테스트 데이터 분할

In [4]:
from sklearn.model_selection import train_test_split

# 학습 데이터와 테스트 데이터 분할 (8:2, stratify=y, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 분할된 데이터의 Class 비율 출력
print("--- 학습 데이터 Class 비율 ---")
print(y_train.value_counts(normalize=True))
print("\n--- 테스트 데이터 Class 비율 ---")
print(y_test.value_counts(normalize=True))

--- 학습 데이터 Class 비율 ---
Class
0    0.953056
1    0.046944
Name: proportion, dtype: float64

--- 테스트 데이터 Class 비율 ---
Class
0    0.953311
1    0.046689
Name: proportion, dtype: float64


# 5. SMOTE 적용

In [5]:
from imblearn.over_sampling import SMOTE

# [SMOTE를 적용하는 이유]
# 신용카드 사기 데이터는 사기 거래(Class=1)가 매우 적은 클래스 불균형 문제를 가지고 있습니다.
# SMOTE를 적용하지 않으면 모델이 다수 클래스인 정상 거래에만 편향되게 학습될 위험이 있습니다.
# SMOTE는 단순히 데이터를 복제하는 것이 아니라 기존 데이터를 바탕으로 가상의 새로운 데이터를 생성하여 모델이 사기 거래의 특징을 더 잘 학습하고 재현율(Recall)을 높일 수 있게 합니다.

# SMOTE 적용 (random_state=42)
smote = SMOTE(random_state=42)
X_train_over, y_train_over = smote.fit_resample(X_train, y_train)

# SMOTE 적용 전후의 사기 거래(Class=1) 건수 출력
print(f"SMOTE 적용 전 사기 거래 건수: {(y_train == 1).sum()}")
print(f"SMOTE 적용 후 사기 거래 건수: {(y_train_over == 1).sum()}")

SMOTE 적용 전 사기 거래 건수: 394
SMOTE 적용 후 사기 거래 건수: 7999


# 6. 모델 학습 및 예측

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, average_precision_score

# 모델 선정 및 학습
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_over, y_train_over)

# 테스트셋에서 예측값(predict)과 예측 확률(predict_proba) 출력
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

# 테스트셋에서 예측값(predict)과 예측 확률(predict_proba) 출력
# 명세서의 지시대로 전체 변수를 출력합니다.
print("--- 예측값 (y_pred) ---")
print(y_pred)

print("\n--- 예측 확률 (y_prob) ---")
print(y_prob)

# 성능 지표 확인 (Precision, Recall, F1-score)
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

# PR-AUC 계산 및 출력
pr_auc = average_precision_score(y_test, y_prob)
print(f"PR-AUC: {pr_auc:.4f}")

--- 예측값 (y_pred) ---
[0 0 0 ... 0 0 0]

--- 예측 확률 (y_prob) ---
[0.02 0.   0.03 ... 0.   0.   0.01]

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      2001
           1       0.95      0.89      0.92        98

    accuracy                           0.99      2099
   macro avg       0.97      0.94      0.96      2099
weighted avg       0.99      0.99      0.99      2099

PR-AUC: 0.9537


# 7. 최종 성능 평가
- 최종 모델의 성능이 목표치(Recall >= 0.80, F1 >= 0.88, PR-AUC >= 0.90)를 모두 달성하였습니다.
- Recall (Class 1): 0.89
- F1-score (Class 1): 0.92
- PR-AUC: 0.9537
-> 모든 지표가 기준을 만족합니다.